# 07 - Train ViT-B/16 Transformer Baseline

This notebook trains **ViT-B/16** as the transformer baseline for the combined skin disease classification project.

Training plan:
- Fixed **12 epochs**.
- **No early stopping**.
- Two-phase fine-tuning:
  - Epochs 1-3: freeze transformer backbone and train classifier head only.
  - Epochs 4-12: unfreeze full ViT and fine-tune all layers.
- Train split uses `WeightedRandomSampler` and live augmentation.
- Validation/test splits stay unchanged and deterministic.
- Best checkpoint is selected using validation macro-F1.
- Final test metrics are saved using the best checkpoint.

## Architecture Summary

ViT-B/16 splits each `224 x 224` image into `16 x 16` patches.

- Patch grid: `14 x 14`
- Total patches: `196`
- Transformer encoder layers: `12`
- Hidden size: `768`
- Attention heads: `12`
- Original ImageNet head: `1000` classes
- Project head: `14` skin disease classes

The classifier head is replaced with `Linear(768 -> 14)`.

In [16]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('/backup/Intern/combine_skindiseaseclassifier_devraj')
EXPECTED_PYTHON = '/backup/Intern/combine_skindiseaseclassifier_devraj/.venv/bin/python'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)

if sys.executable != EXPECTED_PYTHON:
    raise RuntimeError(
        'Wrong notebook kernel selected. Expected: ' + EXPECTED_PYTHON + '\n'
        'Current: ' + sys.executable + '\n'
        'Please switch kernel to: Combine Skin GPU (.venv)'
    )

Project root: /backup/Intern/combine_skindiseaseclassifier_devraj
Python: /backup/Intern/combine_skindiseaseclassifier_devraj/.venv/bin/python


In [17]:
import torch

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA H100 80GB HBM3


## Quick Model Shape Check

This checks that ViT-B/16 outputs `14` logits, one for each disease class.

In [18]:
!cd /backup/Intern/combine_skindiseaseclassifier_devraj && python src/models/vit.py

Input shape : (2, 3, 224, 224)
Output shape: (2, 14)
Total params: 85,809,422
Trainable   : 85,809,422
ViT-B/16 shape test passed.


## Start Training

This trains ViT-B/16 for 12 epochs.

If the pretrained ImageNet weights are not already cached, PyTorch may download them the first time this cell runs.

In [19]:
!cd /backup/Intern/combine_skindiseaseclassifier_devraj && python scripts/train_vit_b16.py \
  --epochs 12 \
  --freeze-epochs 3 \
  --batch-size 32 \
  --phase1-lr 1e-3 \
  --phase2-lr 3e-5 \
  --weight-decay 0.05 \
  --label-smoothing 0.05 \
  --pretrained

ViT-B/16 TRAINING
Project root : /backup/Intern/combine_skindiseaseclassifier_devraj
Data dir     : /backup/Intern/combine_skindiseaseclassifier_devraj/data/splits
Run dir      : /backup/Intern/combine_skindiseaseclassifier_devraj/training_outputs/vit/vit_b_16/vit_b_16_fixed_12ep_20260702_070652
Device       : cuda
GPU          : NVIDIA H100 80GB HBM3
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /home/rohan/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth
100%|███████████████████████████████████████| 330M/330M [00:18<00:00, 18.8MB/s]
Classes      : 14
Train images : 25576
Val images   : 5478
Test images  : 5474
Total params : 85,809,422
Trainable    : 85,809,422

----------------------------------------------------------------------
Phase: head_only | epochs 1-3 | lr=0.001 | trainable=10,766
----------------------------------------------------------------------
Epoch 01/12 | head_only | train_f1=0.6112 | val_f1=0.6417 | val_acc=0.6878 | improved=True
Ep

## Update Overall Model Comparison

Run this after ViT training finishes. It adds ViT to the comparison markdown, CSV, and graphs.

In [20]:
!cd /backup/Intern/combine_skindiseaseclassifier_devraj && python scripts/create_model_comparison_report.py
!cd /backup/Intern/combine_skindiseaseclassifier_devraj && python scripts/update_project_structure.py

Wrote /backup/Intern/combine_skindiseaseclassifier_devraj/reports/model_comparison/model_comparison.csv
Wrote /backup/Intern/combine_skindiseaseclassifier_devraj/markdown_reports/overall_model_comparison_summary.md
Wrote plots under /backup/Intern/combine_skindiseaseclassifier_devraj/reports/model_comparison
Updated: /backup/Intern/combine_skindiseaseclassifier_devraj/PROJECT_STRUCTURE.txt
